# 05. 하이퍼파라미터 최적화

모델(04)과 전처리(03)를 고정하고, 나머지를 **하나씩** 바꾼다.

| 절 | 바꾸는 것 | 왜 중요한가 |
|---|---|---|
| 5-1 | 학습률 | 전이학습에서 영향이 가장 크다 |
| 5-2 | 이미지 크기 | 성능과 **3분 예산**을 동시에 건드린다 |
| 5-3 | 정규화 (weight decay / label smoothing / dropout) | 과적합 억제 |
| 5-4 | 배치 크기 | 속도와 안정성 |
| 5-5 | 최적 조합으로 본 학습 | |

**규칙**: 한 번에 하나만 바꾼다. 모든 판단은 검증셋으로만. 시드는 42 로 고정.

In [ ]:
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

import wbc
wbc.use_korean_font()

cfg = wbc.load_cfg()
MODEL      = cfg['model_name']
PRESET     = cfg['preset']
IMAGE_SIZE = cfg['image_size']
BATCH_SIZE = cfg['batch_size']
SCREEN_EPOCHS = 6
FULL_EPOCHS   = 25
SUBSET     = None
wbc.NUM_WORKERS = 4

print('고정 조건 :', MODEL, '/', PRESET, '/', IMAGE_SIZE, 'px')

## 5-1. 학습률

수업 5장의 원칙: **미세조정은 학습률을 작게.**
너무 크면 사전학습 가중치가 망가지고, 너무 작으면 정해진 에폭 안에 수렴하지 못한다.

In [ ]:
for lr in [1e-3, 5e-4, 3e-4, 1e-4, 5e-5]:
    wbc.run_experiment(f'LR_{lr:g}', model_name=MODEL, preset=PRESET, image_size=IMAGE_SIZE,
                       batch_size=BATCH_SIZE, lr=lr, epochs=SCREEN_EPOCHS,
                       patience=3, seed=42, subset=SUBSET)
    print('-' * 78)

In [ ]:
t = wbc.runs_table(); lrt = t[t.run_id.str.startswith('LR_')].sort_values('lr')
display(lrt[['run_id', 'lr', 'best_epoch', 'val_accuracy', 'val_macro_f1', 'val_loss']].round(5))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].semilogx(lrt.lr, lrt.val_macro_f1, marker='o')
ax[0].set_xlabel('학습률'); ax[0].set_ylabel('검증 macro-F1'); ax[0].grid(alpha=.3); ax[0].set_title('학습률 vs 성능')
ax[1].semilogx(lrt.lr, lrt.val_loss, marker='o', color='tab:red')
ax[1].set_xlabel('학습률'); ax[1].set_ylabel('검증 손실'); ax[1].grid(alpha=.3); ax[1].set_title('학습률 vs 손실')
plt.tight_layout(); plt.show()

BEST_LR = float(lrt.sort_values('val_macro_f1', ascending=False).iloc[0].lr)
print('선정된 학습률:', BEST_LR)

**해석 요령**: 곡선이 봉우리를 그리고 그 꼭짓점이 우리가 고른 값이어야 한다.
꼭짓점이 탐색 구간의 **양 끝**에 있으면 아직 최적점을 못 찾은 것이므로 구간을 넓혀 다시 돌린다.
(예: 1e-3 이 1등이면 3e-3, 5e-3 도 추가로 시도)

## 5-2. 이미지 크기 — 성능과 3분 예산의 맞바꿈

In [ ]:
for size in [128, 160, 224]:
    wbc.run_experiment(f'SZ_{size}', model_name=MODEL, preset=PRESET, image_size=size,
                       batch_size=BATCH_SIZE, lr=BEST_LR, epochs=SCREEN_EPOCHS,
                       patience=3, seed=42, subset=SUBSET)
    print('-' * 78)

In [ ]:
t = wbc.runs_table(); sz = t[t.run_id.str.startswith('SZ_')].sort_values('image_size')
display(sz[['run_id', 'image_size', 'epoch_sec', 'val_accuracy', 'val_macro_f1']].round(4))

fig, ax1 = plt.subplots(figsize=(6.5, 3.6))
ax1.plot(sz.image_size, sz.val_macro_f1, marker='o', color='tab:blue')
ax1.set_xlabel('이미지 크기(px)'); ax1.set_ylabel('검증 macro-F1', color='tab:blue'); ax1.grid(alpha=.3)
ax2 = ax1.twinx(); ax2.plot(sz.image_size, sz.epoch_sec, marker='s', color='tab:red')
ax2.axhline(180, ls='--', c='r'); ax2.set_ylabel('에폭당 시간(초)', color='tab:red')
plt.title('해상도: 성능 vs 시간'); plt.tight_layout(); plt.show()

**여기서 판단**: 224 가 성능이 좋아도 3분을 넘으면 160 을 쓴다.
성능 차이가 0.005 미만이고 시간이 절반이면, 160 이 실용적으로 더 나은 선택이다.

## 5-3. 정규화 — weight decay / label smoothing / dropout

| 기법 | 하는 일 |
|---|---|
| weight decay | 가중치가 커지지 않게 눌러 과적합을 줄인다 |
| label smoothing | 정답을 1.0 대신 0.9 정도로 둬서 과신을 막는다 |
| dropout | 헤드에서 일부 뉴런을 무작위로 꺼 공적응을 막는다 |

In [ ]:
grid = [dict(weight_decay=1e-4, label_smoothing=0.0, dropout=0.0),   # 기준
        dict(weight_decay=1e-2, label_smoothing=0.0, dropout=0.0),
        dict(weight_decay=1e-4, label_smoothing=0.1, dropout=0.0),
        dict(weight_decay=1e-4, label_smoothing=0.0, dropout=0.3),
        dict(weight_decay=1e-2, label_smoothing=0.1, dropout=0.2)]
for i, g in enumerate(grid):
    wbc.run_experiment(f'REG_{i}', model_name=MODEL, preset=PRESET, image_size=IMAGE_SIZE,
                       batch_size=BATCH_SIZE, lr=BEST_LR, epochs=SCREEN_EPOCHS,
                       patience=3, seed=42, subset=SUBSET, **g)
    print('-' * 78)

In [ ]:
t = wbc.runs_table(); reg = t[t.run_id.str.startswith('REG_')].sort_values('val_macro_f1', ascending=False)
display(reg[['run_id', 'weight_decay', 'label_smoothing', 'dropout',
             'val_accuracy', 'val_macro_f1', 'val_loss']].round(5))
b = reg.iloc[0]
BEST_REG = dict(weight_decay=float(b.weight_decay), label_smoothing=float(b.label_smoothing),
                dropout=float(b.dropout))
print('선정된 정규화 설정:', BEST_REG)

## 5-4. 배치 크기

In [ ]:
for bs in [32, 64, 128]:
    try:
        wbc.run_experiment(f'BS_{bs}', model_name=MODEL, preset=PRESET, image_size=IMAGE_SIZE,
                           batch_size=bs, lr=BEST_LR, epochs=SCREEN_EPOCHS,
                           patience=3, seed=42, subset=SUBSET, **BEST_REG)
    except RuntimeError as e:
        print(f'bs={bs} 실패(메모리 부족 가능): {str(e)[:80]}')
    print('-' * 78)

t = wbc.runs_table()
display(t[t.run_id.str.startswith('BS_')][['run_id', 'batch_size', 'epoch_sec',
                                           'val_accuracy', 'val_macro_f1']].round(4))

## 5-5. 최적 조합 저장

지금까지 고른 값을 `config.json` 에 모은다. **06 노트북이 이걸 읽어 본 학습을 한다.**

In [ ]:
bs_t = t[t.run_id.str.startswith('BS_')]
BEST_BS = int(bs_t.sort_values('val_macro_f1', ascending=False).iloc[0].batch_size) if len(bs_t) else BATCH_SIZE
BEST_SIZE = int(sz.sort_values('val_macro_f1', ascending=False).iloc[0].image_size)

cfg = wbc.save_cfg(lr=BEST_LR, image_size=BEST_SIZE, batch_size=BEST_BS,
                   full_epochs=FULL_EPOCHS, **BEST_REG)
print()
print(json.dumps(cfg, ensure_ascii=False, indent=1))

## 05 정리

- 학습률 5종 · 해상도 3종 · 정규화 5조합 · 배치 3종을 **하나씩** 바꿔 비교했다
- 모든 판단은 검증셋으로만 했고, 테스트셋은 아직 열지 않았다
- 최적 설정을 `config.json` 에 저장했다

> **여기서 03 으로 돌아가도 좋다.** 학습률이 크게 바뀌었다면 최적 전처리도 달라질 수 있다.
> 03 → 04 → 05 를 한 바퀴 더 도는 것이 "무한 반복"의 실체다.
> 이미 돌린 실험은 자동으로 건너뛰므로, 새 `run_id` 만 추가로 돌아간다.

→ 다음: **06_최종학습_분류.ipynb**